# Healthcare Data Cleaning & EDA
All data is synthetic. This notebook demonstrates profiling, duplicate handling, missing-value review, validation, and exploratory summaries.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
DATA = ROOT / 'dataset' / 'raw_healthcare_operations.csv'
if not DATA.exists():
    DATA = Path('../dataset/raw_healthcare_operations.csv')
df = pd.read_csv(DATA)
df.head()

## 1. Profile the raw data

In [ ]:
print('Shape:', df.shape)
display(df.info())
display(df.isna().sum().to_frame('missing_values'))
print('Exact duplicate rows:', df.duplicated().sum())

## 2. Remove duplicates and clean types
Missing numeric values are retained as missing; they are not replaced with invented amounts or durations.

In [ ]:
clean = df.drop_duplicates().copy()
clean['admission_date'] = pd.to_datetime(clean['admission_date'], errors='coerce')
for col in ['length_of_stay_days', 'claim_amount_inr']:
    clean[col] = pd.to_numeric(clean[col], errors='coerce')
for col in ['department', 'diagnosis_group', 'case_status', 'follow_up_required']:
    clean[col] = clean[col].astype('string').str.strip().replace('', pd.NA).fillna('Unknown')
clean['invalid_length_of_stay'] = clean['length_of_stay_days'].lt(0).fillna(False)
clean['invalid_claim_amount'] = clean['claim_amount_inr'].lt(0).fillna(False)
clean.head()

## 3. Explore case status and department performance

In [ ]:
status_summary = clean.groupby('case_status').size().rename('case_count').reset_index()
department_summary = clean.groupby('department').agg(
    case_count=('case_id', 'count'),
    avg_length_of_stay=('length_of_stay_days', 'mean'),
    avg_claim_amount=('claim_amount_inr', 'mean')
).reset_index()
display(status_summary)
display(department_summary.round(2))
status_summary.set_index('case_status')['case_count'].plot(kind='bar', title='Cases by Status')
plt.ylabel('Number of cases')
plt.tight_layout()
plt.show()

## 4. Export cleaned data
Unresolved numeric values remain blank so a reviewer can investigate them.

In [ ]:
OUT = Path('output')
OUT.mkdir(exist_ok=True)
clean.to_csv(OUT / 'cleaned_healthcare_operations.csv', index=False)
status_summary.to_csv(OUT / 'status_summary.csv', index=False)
department_summary.to_csv(OUT / 'department_summary.csv', index=False)
print('Saved outputs to', OUT.resolve())